# Datathon CatBoost Baseline

Bu notebook yeni datathon verileri için hazırlanmış CatBoost regression baseline akışıdır.

İçerik:
- Train/test dosyalarını okuma
- Missing value doldurma: eski notebook mantığı korunarak, **train istatistikleri test'e uygulanır**
- Basit feature engineering
- CatBoost ile 5-Fold local RMSE
- Final model + submission üretimi


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

RANDOM_STATE = 42
TARGET = "career_success_score"
ID_COL = "student_id"

TRAIN_PATH = "train.csv"
TEST_PATH = "test_x.csv"
# Colab/Kaggle/VS Code tarafında dosya yolu farklıysa yukarıyı değiştirmen yeterli.


In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test_x.csv")

print("train shape:", train_df.shape)
print("test shape :", test_df.shape)

display(train_df.head())


train shape: (10000, 47)
test shape : (10000, 46)


,student_id,application_year,age,graduation_year,department,university_tier,cgpa,english_exam_score,attendance_rate,failed_courses_count,...,leadership_score,presentation_score,certification_count,bootcamp_count,applications_sent,interviews_attended,hobby,preferred_social_media_platform,career_success_score,mentor_feedback_text
0,STU_000001,2021,21,2021,Computer Engineering,Tier 4,3.17,62.54,77.31,0,...,62.70,58.84,3,1,24,0,photography,LinkedIn,86.78,Proje kalitesi ve makine öğrenimi konusundaki ...
1,STU_000002,2024,20,2024,Computer Engineering,Tier 4,3.24,75.10,87.13,3,...,42.32,40.54,2,0,46,5,reading,YouTube,46.16,Kodlama ve problem çözme becerileri gelişmekte...
2,STU_000003,2024,28,2024,Electrical Electronics Engineering,Tier 4,3.00,68.53,95.64,1,...,47.27,82.56,1,2,46,5,cinema,Reddit,84.08,İleri düzey frontend geliştirme becerileri ile...
3,STU_000004,2019,22,2018,Computer Engineering,Tier 1,2.82,54.85,77.80,2,...,78.69,85.05,2,4,49,7,running,Reddit,89.97,Güçlü bir kodlama yeteneği ve backend geliştir...
4,STU_000005,2026,22,2026,Computer Engineering,Tier 3,2.28,72.25,71.97,1,...,27.22,84.29,1,0,119,13,football,X,92.46,Ürün analizi alanına olan tutkusu ve makine öğ...


In [ ]:
missing_train = train_df.isna().sum()
missing_test = test_df.isna().sum()

missing_summary = pd.DataFrame({
    "train_missing": missing_train,
    "test_missing": missing_test
}).fillna(0).astype(int)

missing_summary = missing_summary[missing_summary.sum(axis=1) > 0].sort_values("train_missing", ascending=False)
display(missing_summary)


,train_missing,test_missing
internship_duration_months,1657,1586
english_exam_score,953,944
github_avg_stars,910,895
open_source_contribution_count,910,895
hr_interview_score,780,759
linkedin_profile_score,668,680
portfolio_score,364,382


## Missing value stratejisi

Eski dosyandaki yaklaşım korunuyor:

- `english_exam_score`: mean
- `linkedin_profile_score`: mean
- `hr_interview_score`: mean
- `portfolio_score`: mean
- `github_avg_stars`: median
- `open_source_contribution_count`: median
- `internship_duration_months`: `internship_count` grubuna göre median

Önemli fark: test tarafında test'in kendi ortalamasını kullanmıyoruz. Train'de öğrenilen mean/median değerleri test'e uygulanıyor. Bu daha doğru ve yarışma akışına daha uygun.


In [ ]:
def add_missing_flags(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Bu flag'ler fill_missing_values'tan ÖNCE alınmalı.
    # Eksik olma bilgisi model için ayrıca sinyal olabilir.
    missing_cols = [
        "english_exam_score",
        "internship_duration_months",
        "portfolio_score",
        "github_avg_stars",
        "open_source_contribution_count",
        "linkedin_profile_score",
        "hr_interview_score"
    ]

    for col in missing_cols:
        if col in df.columns:
            df[f"{col}_was_missing"] = df[col].isna().astype(int)

    return df


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Tarih/yaş ilişkili basit feature'lar
    df["years_since_graduation"] = df["application_year"] - df["graduation_year"]
    df["age_at_graduation"] = df["age"] - df["years_since_graduation"]
    df["is_recent_graduate"] = (df["years_since_graduation"] <= 1).astype(int)

    # Teknik skor özetleri
    technical_skill_cols = [
        "coding_score", "problem_solving_score", "data_structures_score",
        "sql_score", "machine_learning_score", "backend_score",
        "frontend_score", "cloud_score", "devops_score"
    ]

    df["technical_skill_mean"] = df[technical_skill_cols].mean(axis=1)
    df["technical_skill_std"] = df[technical_skill_cols].std(axis=1)
    df["technical_skill_min"] = df[technical_skill_cols].min(axis=1)
    df["technical_skill_max"] = df[technical_skill_cols].max(axis=1)
    df["technical_skill_range"] = df["technical_skill_max"] - df["technical_skill_min"]

    # Data/AI ve software odaklı teknik alt skorlar
    df["data_ai_score"] = df[
        [
            "sql_score",
            "machine_learning_score",
            "problem_solving_score",
            "data_structures_score",
            "coding_score"
        ]
    ].mean(axis=1)

    df["software_engineering_score"] = df[
        [
            "backend_score",
            "frontend_score",
            "cloud_score",
            "devops_score",
            "coding_score",
            "data_structures_score"
        ]
    ].mean(axis=1)

    # Soft skill özeti
    soft_skill_cols = [
        "communication_score",
        "teamwork_score",
        "leadership_score",
        "presentation_score",
        "linkedin_profile_score",
        "cv_quality_score",
        "hr_interview_score"
    ]

    df["soft_skill_mean"] = df[soft_skill_cols].mean(axis=1)
    df["soft_skill_std"] = df[soft_skill_cols].std(axis=1)

    # Deneyim/aktivite özeti
    experience_cols = [
        "real_client_project_count",
        "internship_count",
        "freelance_project_count",
        "hackathon_count",
        "certification_count",
        "bootcamp_count"
    ]

    df["experience_total"] = df[experience_cols].sum(axis=1)

    df["project_portfolio_score"] = (
        df["project_quality_score"]
        + df["portfolio_score"]
        + df["real_client_project_count"] * 5
        + df["freelance_project_count"] * 3
        + df["github_repo_count"] * 0.5
        + df["github_avg_stars"] * 0.5
        + df["open_source_contribution_count"] * 2
    )

    df["experience_score"] = (
        df["internship_count"] * 10
        + df["internship_duration_months"] * 2
        + df["freelance_project_count"] * 5
        + df["real_client_project_count"] * 7
    )

    df["competition_score"] = (
        df["hackathon_count"] * 3
        + df["hackathon_awards"] * 10
    )

    df["learning_activity_score"] = (
        df["certification_count"] * 2
        + df["bootcamp_count"] * 5
    )

    # Başvuru -> mülakat oranı
    df["interview_conversion_rate"] = (
        df["interviews_attended"] / (df["applications_sent"] + 1)
    )

    df["applications_without_interview"] = (
        df["applications_sent"] - df["interviews_attended"]
    )

    df["applications_per_year_after_grad"] = (
        df["applications_sent"] / (df["years_since_graduation"] + 1)
    )

    df["interview_per_year_after_grad"] = (
        df["interviews_attended"] / (df["years_since_graduation"] + 1)
    )

    df["interview_score_mean"] = df[
        ["technical_interview_score", "hr_interview_score"]
    ].mean(axis=1)

    df["weighted_interview_score"] = (
        df["technical_interview_score"] * 0.6
        + df["hr_interview_score"] * 0.4
    )

    df["interview_success_score"] = (
        df["interview_conversion_rate"] * df["weighted_interview_score"]
    )

    # Interaction feature'lar
    df["technical_x_project"] = (
        df["technical_skill_mean"] * df["project_portfolio_score"]
    )

    df["technical_x_experience"] = (
        df["technical_skill_mean"] * df["experience_score"]
    )

    df["soft_x_interview"] = (
        df["soft_skill_mean"] * df["weighted_interview_score"]
    )

    df["github_impact_score"] = (
        df["github_repo_count"]
        + df["github_avg_stars"] * 2
        + df["open_source_contribution_count"] * 3
    )

    df["internship_months_per_internship"] = (
        df["internship_duration_months"] / (df["internship_count"] + 1)
    )

    # Text'i modele direkt vermek yerine başlangıç için uzunluk feature'ları alıyoruz.
    # CatBoost text feature destekliyor ama localde daha yavaş çalışabilir.
    df["mentor_feedback_len"] = (
        df["mentor_feedback_text"]
        .fillna("")
        .astype(str)
        .str.len()
    )

    df["mentor_feedback_word_count"] = (
        df["mentor_feedback_text"]
        .fillna("")
        .astype(str)
        .str.split()
        .str.len()
    )

    # Sonsuz değer temizliği
    df = df.replace([np.inf, -np.inf], np.nan)

    return df


In [ ]:
def fill_missing_values(train: pd.DataFrame, test: pd.DataFrame):
    train = train.copy()
    test = test.copy()

    # 1) internship_duration_months: internship_count grubuna göre median
    group_medians = train.groupby("internship_count")["internship_duration_months"].median()
    global_median = train["internship_duration_months"].median()

    for df in [train, test]:
        df["internship_duration_months"] = (
            df["internship_duration_months"]
            .fillna(df["internship_count"].map(group_medians))
            .fillna(global_median)
        )

    # 2) Mean ile doldurulacak kolonlar
    mean_fill_cols = [
        "english_exam_score",
        "linkedin_profile_score",
        "hr_interview_score",
        "portfolio_score"
    ]

    for col in mean_fill_cols:
        fill_value = train[col].mean()
        train[col] = train[col].fillna(fill_value)
        test[col] = test[col].fillna(fill_value)

    # 3) Median ile doldurulacak kolonlar
    median_fill_cols = [
        "github_avg_stars",
        "open_source_contribution_count"
    ]

    for col in median_fill_cols:
        fill_value = train[col].median()
        train[col] = train[col].fillna(fill_value)
        test[col] = test[col].fillna(fill_value)

    # 4) Kategorik/text kolonlarda boşluk varsa Unknown
    object_cols = train.select_dtypes(include="object").columns.tolist()
    for col in object_cols:
        if col != ID_COL:
            train[col] = train[col].fillna("Unknown").astype(str)
            test[col] = test[col].fillna("Unknown").astype(str)

    return train, test


In [ ]:
# 1) Önce missing flag'leri al
train_tmp = add_missing_flags(train_df)
test_tmp = add_missing_flags(test_df)

# 2) Sonra eksik değerleri doldur
train_tmp, test_tmp = fill_missing_values(train_tmp, test_tmp)

# 3) En son yeni feature'ları üret
# Önemli: project_portfolio_score, experience_score, weighted_interview_score gibi feature'lar
# eksik kolonları kullandığı için add_features, fill_missing_values'tan SONRA çalışmalı.
train_fe = add_features(train_tmp)
test_fe = add_features(test_tmp)

print("Train missing after fill:", train_fe.isna().sum().sum())
print("Test missing after fill :", test_fe.isna().sum().sum())

print("Train shape after features:", train_fe.shape)
print("Test shape after features :", test_fe.shape)


Train missing after fill: 0
Test missing after fill : 0


In [ ]:
# Başlangıç baseline'da text kolonunu direkt kullanmıyoruz; uzunluk feature'ları zaten eklendi.
DROP_COLS = [ID_COL, TARGET, "mentor_feedback_text"]

features = [col for col in train_fe.columns if col not in DROP_COLS]

X = train_fe[features]
y = train_fe[TARGET]
X_test = test_fe[features]

cat_features = X.select_dtypes(include="object").columns.tolist()
cat_feature_indices = [X.columns.get_loc(col) for col in cat_features]

print("Feature count:", len(features))
print("Categorical features:", cat_features)


Feature count: 53
Categorical features: ['department', 'university_tier', 'target_role', 'hobby', 'preferred_social_media_platform']


In [ ]:
cat_params = {
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "iterations": 1500,
    "learning_rate": 0.03,
    "depth": 6,
    "l2_leaf_reg": 5,
    "random_seed": RANDOM_STATE,
    "od_type": "Iter",
    "od_wait": 100,
    "verbose": 200,
    "allow_writing_files": False
}


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

oof_preds = np.zeros(len(train_fe))
test_preds = np.zeros(len(test_fe))
fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y), start=1):
    print(f"========== Fold {fold} ==========")

    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = CatBoostRegressor(**cat_params)
    model.fit(
        X_train, y_train,
        cat_features=cat_feature_indices,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )

    valid_pred = model.predict(X_valid)
    oof_preds[valid_idx] = valid_pred

    fold_rmse = mean_squared_error(y_valid, valid_pred) ** 0.5
    fold_scores.append(fold_rmse)
    print(f"Fold {fold} RMSE: {fold_rmse:.5f}")

    test_preds += model.predict(X_test) / kf.n_splits

oof_rmse = mean_squared_error(y, oof_preds) ** 0.5
print("========== CV RESULT ==========")
print(f"Fold RMSE values: {[round(s, 5) for s in fold_scores]}")
print(f"Mean RMSE: {np.mean(fold_scores):.5f}")
print(f"Std RMSE : {np.std(fold_scores):.5f}")
print(f"OOF RMSE : {oof_rmse:.5f}")


========== Fold 1 ==========
0:	learn: 15.0028555	test: 14.9861831	best: 14.9861831 (0)	total: 121ms	remaining: 3m 1s
200:	learn: 8.9881330	test: 9.5012915	best: 9.5012915 (200)	total: 2.8s	remaining: 18.1s
400:	learn: 8.2966852	test: 9.2038274	best: 9.2038274 (400)	total: 5.4s	remaining: 14.8s
600:	learn: 7.8857885	test: 9.1084924	best: 9.1083376 (597)	total: 8.14s	remaining: 12.2s
800:	learn: 7.5181720	test: 9.0885424	best: 9.0872355 (761)	total: 10.7s	remaining: 9.31s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 9.087235483
bestIteration = 761

Shrink model to first 762 iterations.
Fold 1 RMSE: 9.08724
========== Fold 2 ==========
0:	learn: 14.9590506	test: 15.2154857	best: 15.2154857 (0)	total: 10.4ms	remaining: 15.6s
200:	learn: 8.9209805	test: 9.7083210	best: 9.7083210 (200)	total: 2.58s	remaining: 16.7s
400:	learn: 8.2499372	test: 9.4103536	best: 9.4103508 (399)	total: 5.11s	remaining: 14s
600:	learn: 7.8223222	test: 9.3229062	best: 9.3229062 (600)	total: 7

In [ ]:
# Tahminleri target aralığına sıkıştırmak genelde bu veri için mantıklı: target 0-100 aralığında.
test_preds_clipped = np.clip(test_preds, 0, 100)

submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET: test_preds_clipped
})

submission.to_csv("catboost_submission.csv", index=False)
display(submission.head())
print("Saved: catboost_submission.csv")


,student_id,career_success_score
0,STU_010001,58.054123
1,STU_010002,72.070891
2,STU_010003,72.610683
3,STU_010004,95.926693
4,STU_010005,78.564187


Saved: catboost_submission.csv


## Alternatif: Text feature'ı CatBoost'a direkt vermek

`mentor_feedback_text` kolonu güçlü olabilir ama CatBoost text processing daha yavaş çalışabilir. Baseline oturduktan sonra aşağıdaki alternatifi deneyebilirsin:

- `mentor_feedback_text` kolonunu `DROP_COLS` listesinden çıkar
- `text_features = ["mentor_feedback_text"]` tanımla
- `model.fit(...)` içinde `text_features=text_features` kullanmak yerine `Pool` ile eğitim yap

Başlangıç için yukarıdaki sürüm daha hızlı ve daha az sorun çıkarır.
